In [1]:
!pip install scikit-learn pandas joblib

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import accuracy_score, classification_report
import joblib

In [3]:
# Upload dataset.csv in Colab first
data = pd.read_csv("dataset.csv")

# Remove comment rows if any
data = data[~data['login_hour'].astype(str).str.contains("#", na=False)]

data.head(10)

,login_hour,session_duration,transactions_count,amount_transferred,download_count,ip_change,new_payee_added,failed_attempts
0,10,30,2,5000,1,0,0,0
1,11,40,3,7000,1,0,0,0
2,9,25,1,2000,0,0,0,0
3,14,50,4,8000,1,0,0,0
4,12,35,2,6000,1,0,0,0
5,13,45,3,7500,1,0,0,0
6,10,28,2,4500,0,0,0,0
7,15,55,5,9000,2,0,0,0
8,11,38,3,6500,1,0,0,0
9,9,20,1,3000,0,0,0,0


In [4]:
# Create labels manually
labels = []

for i in range(len(data)):
    if i < 20:   # first 20 rows = normal
        labels.append(1)
    else:        # last rows = anomaly
        labels.append(-1)

data['label'] = labels

In [5]:
X = data.drop("label", axis=1)
y = data["label"]

In [6]:
# Train only on normal data
X_train = X[y == 1]

model = IsolationForest(contamination=0.2, random_state=42)
model.fit(X_train)

IsolationForest(contamination=0.2, random_state=42)

In [7]:
y_pred = model.predict(X)

In [8]:
accuracy = accuracy_score(y, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.84


In [9]:
print(classification_report(y, y_pred))

              precision    recall  f1-score   support

          -1       0.56      1.00      0.71         5
           1       1.00      0.80      0.89        20

    accuracy                           0.84        25
   macro avg       0.78      0.90      0.80        25
weighted avg       0.91      0.84      0.85        25



In [10]:
# Format:
# [login_hour, session_duration, transactions_count, amount_transferred,
#  download_count, ip_change, new_payee_added, failed_attempts]

user_input = np.array([[10, 30, 2, 5000, 1, 0, 0, 0]])

In [11]:
prediction = model.predict(user_input)
score = model.decision_function(user_input)

print("Prediction:", prediction)
print("Anomaly Score:", score)

Prediction: [1]
Anomaly Score: [0.07680538]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


In [12]:
import random
import pandas as pd

data_list = []

# 🔹 Generate NORMAL data (200 rows)
for _ in range(200):
    data_list.append([
        random.randint(9,18),      # login_hour
        random.randint(20,60),     # session_duration
        random.randint(1,5),       # transactions
        random.randint(1000,10000),# amount
        random.randint(0,2),       # downloads
        0,                         # ip_change
        0,                         # new_payee
        random.randint(0,1)        # failed attempts
    ])

# 🔴 Generate ANOMALY data (40 rows)
for _ in range(40):
    data_list.append([
        random.randint(1,5),
        random.randint(200,400),
        random.randint(10,20),
        random.randint(100000,500000),
        random.randint(5,15),
        1,
        1,
        random.randint(2,5)
    ])

# Create DataFrame
columns = [
    "login_hour","session_duration","transactions_count",
    "amount_transferred","download_count","ip_change",
    "new_payee_added","failed_attempts"
]

data = pd.DataFrame(data_list, columns=columns)

data.head(10)

,login_hour,session_duration,transactions_count,amount_transferred,download_count,ip_change,new_payee_added,failed_attempts
0,18,54,4,7785,0,0,0,1
1,17,26,2,7783,1,0,0,1
2,10,35,2,4043,2,0,0,1
3,17,52,3,7601,1,0,0,1
4,18,41,2,5506,1,0,0,0
5,14,31,3,4219,2,0,0,1
6,11,23,3,1492,2,0,0,1
7,18,39,5,7290,1,0,0,0
8,14,42,2,9095,2,0,0,0
9,14,52,3,4241,0,0,0,1


In [13]:
labels = [1]*200 + [-1]*40
data["label"] = labels

In [14]:
from sklearn.ensemble import IsolationForest

X = data.drop("label", axis=1)
y = data["label"]

# Train only on normal
X_train = X[y == 1]

model = IsolationForest(
    n_estimators=200,      # more trees
    contamination=0.15,    # tuned
    random_state=42
)

model.fit(X_train)

IsolationForest(contamination=0.15, n_estimators=200, random_state=42)

In [15]:
y_pred = model.predict(X)

from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y, y_pred)
print("New Accuracy:", accuracy)

New Accuracy: 0.875


In [16]:
from sklearn.model_selection import train_test_split

# Features & labels
X = data.drop("label", axis=1)
y = data["label"]

# Split into train + test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train only on NORMAL data
X_train = X_train_full[y_train_full == 1]

In [17]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=200,
    contamination=0.15,
    random_state=42
)

model.fit(X_train)

IsolationForest(contamination=0.15, n_estimators=200, random_state=42)

In [18]:
y_pred = model.predict(X_test)

In [19]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 0.6458333333333334

Classification Report:
              precision    recall  f1-score   support

          -1       0.35      1.00      0.51         9
           1       1.00      0.56      0.72        39

    accuracy                           0.65        48
   macro avg       0.67      0.78      0.62        48
weighted avg       0.88      0.65      0.68        48



In [20]:
import random
import pandas as pd

data_list = []

# ✅ NORMAL DATA (300 rows)
for _ in range(300):
    data_list.append([
        random.randint(9,18),        # login_hour
        random.randint(20,60),       # session_duration
        random.randint(1,5),         # transactions
        random.randint(1000,10000),  # amount
        random.randint(0,2),         # downloads
        0,                           # ip_change
        0,                           # new_payee
        random.randint(0,1),         # failed attempts
        1                            # label
    ])

# 🚨 ANOMALY DATA (80 rows) → make VERY different
for _ in range(80):
    data_list.append([
        random.randint(1,5),
        random.randint(300,500),     # more extreme
        random.randint(10,25),
        random.randint(150000,600000),
        random.randint(5,20),
        1,
        1,
        random.randint(3,7),
        -1
    ])

columns = [
    "login_hour","session_duration","transactions_count",
    "amount_transferred","download_count","ip_change",
    "new_payee_added","failed_attempts","label"
]

data = pd.DataFrame(data_list, columns=columns)

# ✅ Save dataset
data.to_csv("improved_dataset.csv", index=False)

print("Dataset created:", data.shape)
data.head()

Dataset created: (380, 9)


,login_hour,session_duration,transactions_count,amount_transferred,download_count,ip_change,new_payee_added,failed_attempts,label
0,12,51,4,9998,0,0,0,1,1
1,10,29,3,3742,0,0,0,0,1
2,18,32,4,5981,2,0,0,0,1
3,13,60,1,9858,2,0,0,1,1
4,16,45,1,3361,1,0,0,0,1


In [21]:
from sklearn.model_selection import train_test_split

X = data.drop("label", axis=1)
y = data["label"]

# Separate normal and anomaly
X_normal = X[y == 1]
X_anomaly = X[y == -1]

# Split only normal
X_train, X_test_normal = train_test_split(X_normal, test_size=0.2, random_state=42)

# Combine test data
X_test = pd.concat([X_test_normal, X_anomaly])
y_test = [1]*len(X_test_normal) + [-1]*len(X_anomaly)

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [23]:
from sklearn.ensemble import IsolationForest

model = IsolationForest(
    n_estimators=300,     # more trees
    contamination=0.2,    # tuned
    random_state=42
)

model.fit(X_train)

IsolationForest(contamination=0.2, n_estimators=300, random_state=42)

In [24]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Final Test Accuracy:", accuracy)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Final Test Accuracy: 0.8571428571428571

Classification Report:

              precision    recall  f1-score   support

          -1       0.80      1.00      0.89        80
           1       1.00      0.67      0.80        60

    accuracy                           0.86       140
   macro avg       0.90      0.83      0.84       140
weighted avg       0.89      0.86      0.85       140



In [25]:
# Predict on training data
y_train_pred = model.predict(X_train)

from sklearn.metrics import accuracy_score

train_accuracy = accuracy_score([1]*len(X_train), y_train_pred)

print("Training Accuracy:", train_accuracy)

Training Accuracy: 0.8


In [26]:
# Predict on test data
y_test_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_test_pred)

print("Test Accuracy:", test_accuracy)

Test Accuracy: 0.8571428571428571


In [27]:
from sklearn.metrics import classification_report

print("\nTraining Report:")
print(classification_report([1]*len(X_train), y_train_pred))

print("\nTesting Report:")
print(classification_report(y_test, y_test_pred))


Training Report:
              precision    recall  f1-score   support

          -1       0.00      0.00      0.00         0
           1       1.00      0.80      0.89       240

    accuracy                           0.80       240
   macro avg       0.50      0.40      0.44       240
weighted avg       1.00      0.80      0.89       240


Testing Report:
              precision    recall  f1-score   support

          -1       0.80      1.00      0.89        80
           1       1.00      0.67      0.80        60

    accuracy                           0.86       140
   macro avg       0.90      0.83      0.84       140
weighted avg       0.89      0.86      0.85       140



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
